In [ ]:
library(Seurat)
library(ggplot2)
library(pheatmap)
library(dplyr)
library(RColorBrewer)
library(clustree)
library(reshape2)
getwd()
dir.create("figures_10xMouse_PBMC")
dir.create("data_10xMouse_PBMC")
dataset_id <- "10xMouse_PBMC"
sample_id <- "10k"
colorPBMC <- "#81B29A"


In [ ]:
getwd()

conversionTable <- read.table("annotation/annotation_mm10_conversion_withAge.tsv") # created with annotation_scripts/create_annotations_mouse.Rmd
head(conversionTable)

# Import SoloTE results and split genes and TEs

In [ ]:
SoloTE_path <- paste0("/mnt/volume_1p5T/results/SoloTEout/", dataset_id, "/", sample_id, "/", sample_id,"_SoloTE_output/", sample_id,"_legacytes_MATRIX")
STAR_path <- paste0("/mnt/TEresults/snakemake_results/results/STARoutdir/",dataset_id,"/",sample_id,"/best_Solo.out/Gene")
filteredBarcodes <- read.table(paste0(STAR_path, "/filtered/barcodes.tsv"))$V1 # read barcodes seleced by STARsolo

In [ ]:
# remove cells annotated as multiplets

# import cell assignment done by 10x
mouse_assignment <- read.csv("data_10xMouse_PBMC/SC3_v3_NextGem_DI_CellPlex_Mouse_PBMC_10K_Multiplex_multiplexing_analysis_assignment_confidence_table.csv")
mouse_assignment$Barcodes <- sapply(strsplit(mouse_assignment$Barcodes, split="-"), "[", 1)

multiplets <- mouse_assignment$Barcodes[mouse_assignment$Assignment=="Multiplet"]


filteredBarcodes <- setdiff(filteredBarcodes, multiplets)


In [ ]:
legacyTEmatrix <- Seurat::ReadMtx(mtx = paste0(SoloTE_path, "/matrix.mtx"), 
                              cells = paste0(SoloTE_path, "/barcodes.tsv"), 
                              features = paste0(SoloTE_path, "/features.tsv")) # read matrix
legacyTEmatrix <- legacyTEmatrix[,filteredBarcodes]

# select TEs
TEs <- grep("SoloTE", rownames(legacyTEmatrix), value = T)
locusTEs <- grep("chr", TEs, value = T)
# subset the matrix keeping only TEs
TEmatrix <- legacyTEmatrix[locusTEs,]
# remove "SoloTE" from the name of the TEs
rownames(TEmatrix) <- gsub("SoloTE\\|", "", rownames(TEmatrix))

rownames(TEmatrix) <- gsub("\\|", "-", rownames(TEmatrix))
rownames(TEmatrix) <- gsub("\\_", "-", rownames(TEmatrix))
rownames(TEmatrix) <- gsub("\\?", "", rownames(TEmatrix))

#table(rownames(TEmatrix) %in% conversionTable$soloteID)
# transform into Stellarscope IDs
#rownames(TEmatrix) <- conversionTable$stellarscopeID[match(rownames(TEmatrix), conversionTable$soloteID)]
nCells <- ncol(TEmatrix)
thrMinCells <- round(nCells * 0.05)

# create Seurat object with shallow filtering of TEs expressed in at least 5% of cells and cells expressing at least 50 TEs 
objTE <- Seurat::CreateSeuratObject(TEmatrix, project = "PBMC",
                                    min.cells = thrMinCells, min.features = 20) 
objTE

In [ ]:
# select genes
genes <- setdiff(rownames(legacyTEmatrix), TEs)
# subset the matrix keeping only genes
Genematrix <- legacyTEmatrix[genes,]
# create Seurat object with shallow filtering of genes expressed in at least 50 cells and cells expressing at least 100 genes 
Gene <- Seurat::CreateSeuratObject(Genematrix, project = "PBMC", 
                          min.cells = thrMinCells, min.features = 200) 

#filteredBarcodes <- read.table(paste0(STAR_path, "/filtered/barcodes.tsv"))$V1 # read barcodes seleced by STARsolo


objTE_SoloTE <- objTE[,filteredBarcodes] # keep only filtered barcodes
objGenes_SoloTE <- Gene[,filteredBarcodes] # keep only filtered barcodes

objTE_SoloTE
objGenes_SoloTE

# QC

In [ ]:
# QC TEs

options(repr.plot.width=7, repr.plot.height=6)

objTE_SoloTE@meta.data$nCount_TE <- objTE_SoloTE@meta.data$nCount_RNA 
objTE_SoloTE@meta.data$nFeature_TE <- objTE_SoloTE@meta.data$nFeature_RNA 

# # Visualize QC metrics as a violin plot
VlnPlot(objTE_SoloTE, features = c("nCount_RNA", "nFeature_RNA"), ncol = 2, 
        pt.size = 0, alpha = 0.5) 
summary(objTE_SoloTE$nCount_RNA)
summary(objTE_SoloTE$nFeature_RNA)

In [ ]:
# QC genes
options(repr.plot.width=8, repr.plot.height=5)

VlnPlot(objGenes_SoloTE, features = c( "nCount_RNA", "nFeature_RNA" ), ncol = 2, 
        pt.size = 0.05, alpha = 0.5, group.by = "orig.ident") 

summary(objGenes_SoloTE$nFeature_RNA)
summary(objGenes_SoloTE$nCount_RNA)

# Genes

In [ ]:
objGenes_SoloTE <- NormalizeData(objGenes_SoloTE, normalization.method = "LogNormalize", scale.factor = 10000)

objGenes_SoloTE <- FindVariableFeatures(objGenes_SoloTE, selection.method = "vst", nfeatures = 4000)

# Identify the 10 most highly variable genes
top10 <- head(VariableFeatures(objGenes_SoloTE), 10)

# plot variable features with and without labels
plot1 <- VariableFeaturePlot(objGenes_SoloTE)
plot2 <- LabelPoints(plot = plot1, points = top10, repel = TRUE)
plot2

In [ ]:
gc()
all.genes <- rownames(objGenes_SoloTE)
objGenes_SoloTE <- ScaleData(objGenes_SoloTE) # on hvgs

objGenes_SoloTE <- RunPCA(objGenes_SoloTE, features = VariableFeatures(object = objGenes_SoloTE))


DimPlot(objGenes_SoloTE, reduction = "pca") + NoLegend()

ElbowPlot(objGenes_SoloTE)


In [ ]:
objGenes_SoloTE <- FindNeighbors(objGenes_SoloTE, dims = 1:10, k.param = 20)
objGenes_SoloTE <- FindClusters(objGenes_SoloTE, resolution = 1)
objGenes_SoloTE <- RunUMAP(objGenes_SoloTE, dims = 1:10)
DimPlot(objGenes_SoloTE, reduction = "umap")

# Celltype annotation

## Plot markers

In [ ]:
mouse_pbmc_markers_broad <- list(
  "T_cells"       = c("Cd3d", "Cd3e", "Cd3g"),
  "NK_cells"      = c("Nkg7"),
  "B_cells"       = c("Cd79a", "Cd79b", "Ms4a1"),
  "Monocytes"     = c("Lyz2", "Csf1r", "Ccr2", "Fcgr4"),
  "DCs"           = c("Flt3", "H2-Ab1"),
  "Platelets"     = c("Ppbp", "Pf4"),
  "Neutrophils"   = c("S100a8", "S100a9", "Ly6g", "Mpo")
)

In [ ]:
options(repr.plot.width=15, repr.plot.height=6)

for(celltype in names(mouse_pbmc_markers_broad)){
    print(mouse_pbmc_markers_broad[celltype])
    show(FeaturePlot(objGenes_SoloTE, reduction = "umap", 
        features=mouse_pbmc_markers_broad[[celltype]], ncol=3) & 
        theme(text=element_text(size=20), plot.title = element_text(hjust=0.5)) )
}

In [ ]:
options(repr.plot.width=7, repr.plot.height=6)

DimPlot(objGenes_SoloTE, reduction = "umap", label=TRUE)

In [ ]:
# 1. Define the mapping from cluster number -> cell type
#    Names are cluster IDs (as they appear in Idents(seu) or seu$seurat_clusters),
#    values are the cell type labels.
cluster_to_celltype <- c(
  "0" = "Platelets",
  "1" = "B_cells",
  "2" = "B_cells",
  "3" = "T_cells",#"T_CD8",
  "4" = "T_cells",#"T_CD4",
  "5" = "B_cells",
  "6" = "T_cells",#"T_CD8",
  "7" = "Monocytes",
  "8" = "Dendritic Cells",
  "9" = "Dendritic Cells",
  "10" = "Dendritic Cells",
  "11" = "Unknown",
  "12" = "Neutrophils",
  "13" = "Platelets",
  "14" = "Monocytes",
  "15" = "Natural Killers",
  "16" = "B_cells"
 )

# 2. Make sure the cluster identities are set as the active idents
#    (adjust "seurat_clusters" if your cluster column has a different name)
Idents(objGenes_SoloTE) <- "seurat_clusters"

# 3. Map cluster labels to cell types and store as a new metadata column
objGenes_SoloTE$celltype <- as.vector(cluster_to_celltype[as.character(Idents(objGenes_SoloTE))])

# 4. (Optional) set celltype as the active identity for downstream plotting
Idents(objGenes_SoloTE) <- "celltype"

# 5. Sanity check
table(objGenes_SoloTE$celltype, objGenes_SoloTE$seurat_clusters)

In [ ]:
options(repr.plot.width=7, repr.plot.height=6)

DimPlot(objGenes_SoloTE, reduction = "umap", label=TRUE)

In [ ]:
# Replace with your actual ambiguous cluster IDs
Idents(objGenes_SoloTE) <- "seurat_clusters"

# Get top markers for each ambiguous cluster vs. all others
markers <- FindAllMarkers(
  objGenes_SoloTE,
  only.pos = TRUE,                # upregulated genes only, easier to interpret
  min.pct = 0.25,                 # gene must be in ≥25% of cells in the cluster
  logfc.threshold = 0.25
)

head(markers)


In [ ]:

# Top 10 markers per cluster, ranked by fold change
library(dplyr)

top_markers <- markers[markers$cluster == 11,] %>%
  slice_max(order_by = avg_log2FC, n = 20)

print(top_markers$gene)

In [ ]:

options(repr.plot.width=6, repr.plot.height=5)


DimPlot(objGenes_SoloTE, reduction = "umap",label = TRUE) + NoLegend()

In [ ]:
#saveRDS(objGenes_SoloTE, file=paste0("data_", dataset_id, "/objGenes_SoloTE.RDS"))

objGenes_SoloTE <- readRDS(file=paste0("data_", dataset_id, "/objGenes_SoloTE.RDS"))

In [ ]:
write.table(objGenes_SoloTE$celltype, quote=FALSE, col.names = FALSE, sep = "\t",
    file= paste0("data_",dataset_id,"/celltype_annotation.tsv"))


# TEs 

In [ ]:

### Normalize

objTE_SoloTE <- NormalizeData(objTE_SoloTE, normalization.method = "LogNormalize", scale.factor = 10000)

objTE_SoloTE <- FindVariableFeatures(objTE_SoloTE, selection.method = "vst", nfeatures = 4000)

# Identify the 10 most highly variable genes
top10 <- head(VariableFeatures(objTE_SoloTE), 10)

# plot variable features with and without labels
plot1 <- VariableFeaturePlot(objTE_SoloTE)
plot2 <- LabelPoints(plot = plot1, points = top10, repel = TRUE)
plot2

In [ ]:

gc()
all.genes <- rownames(objTE_SoloTE)
objTE_SoloTE <- ScaleData(objTE_SoloTE) # on hvgs

grep("MERVL", all.genes, value = T)[1:50]


In [ ]:

objTE_SoloTE <- RunPCA(objTE_SoloTE, features = VariableFeatures(object = objTE_SoloTE))


DimPlot(objTE_SoloTE, reduction = "pca") + NoLegend()

ElbowPlot(objTE_SoloTE)

In [ ]:
objTE_SoloTE <- FindNeighbors(objTE_SoloTE, dims = 1:10, k.param = 20)
objTE_SoloTE <- FindClusters(objTE_SoloTE, resolution = 1)
objTE_SoloTE <- RunUMAP(objTE_SoloTE, dims = 1:10)
DimPlot(objTE_SoloTE, reduction = "umap") +   
    theme_void() +
    theme(text=element_text(size=20)) 

# Checkpoint

In [ ]:
saveRDS(objTE_SoloTE, paste0("data_", dataset_id, "/soloTE_", dataset_id, "_seuratObj.RDS"))
#objTE_SoloTE <- readRDS(paste0("data_", dataset_id, "/soloTE_", dataset_id, "_seuratObj.RDS"))

In [ ]:
options(repr.plot.width=9.5, repr.plot.height=7)
library(ggpubr)
library(scico)

DimPlot(objTE_SoloTE, reduction = "umap", 
        shuffle=T, pt.size = 1) + 
        ggtitle("TE-derived clusters") +
  scale_color_manual(values= scico(length(unique(objTE_SoloTE$seurat_clusters)), palette = 'managua')) +
  theme_pubr() +
  theme(text=element_text(size=20), plot.title = element_text(hjust=0.5)) 
ggsave(paste0("figures_",dataset_id,"/umap_TEs_locus_clusters_SoloTE_scico.pdf"), device = "pdf", width=9.5, height=7)

In [ ]:
scico(30, palette = 'managua')
#>  [1] "#190C64" "#1C176B" "#202272" "#212B79" "#243580" "#263D86" "#29478B"
#>  [8] "#2C5091" "#2F5996" "#33619A" "#37699D" "#3D71A0" "#4479A1" "#4D81A2"
#> [15] "#5688A4" "#608EA2" "#6B94A1" "#77999F" "#839E9C" "#90A198" "#9BA495"
#> [22] "#A9A895" "#B7AD96" "#C7B59C" "#D7BEA6" "#E5C9B3" "#F0D4C3" "#F7DFD3"
#> [29] "#FCE9E3" "#FEF2F2"

# Compare cluters

In [ ]:
clustersResList <- list()
identical(Cells(objGenes_SoloTE), Cells(objTE_SoloTE)) # TRUE, same cells in same order

for(res in seq(0.5, 2, by=0.1)){
    print(res)
    
    clusters_genes <- FindClusters(objGenes_SoloTE, resolution = res)
    clusters_TEs <- FindClusters(objTE_SoloTE, resolution = res)

    clustersResList[[as.character(res)]] <- cbind(clusters_genes$seurat_clusters, clusters_TEs$seurat_clusters)
    #
}


In [ ]:
library(mclust)
library(viridis)

resolutions <- seq(0.5, 2.0, by = 0.1)

ari_matrix <- matrix(data=NA, nrow=length(resolutions), ncol=length(resolutions))
colnames(ari_matrix) <- paste0("gene_r",resolutions)
rownames(ari_matrix) <- paste0("TE_r",resolutions)

for(r_gene in resolutions){
    for(r_TE in resolutions){
        ari <- adjustedRandIndex(clustersResList[[as.character(r_gene)]][,1],
                    clustersResList[[as.character(r_TE)]][,2])
        ari_matrix[paste0("TE_r",r_TE),paste0("gene_r",r_gene)] <- ari
    }
}
ari_matrix

pheatmap(ari_matrix, display_numbers = T, color = mako(100, alpha = 1, begin = 0, end = 1, direction = 1)[],
         border_color = NA, number_format = "%.3f", number_color="black",
         cluster_rows = FALSE, cluster_columns = FALSE,
         cellwidth = 25, cellheight = 25)

         
# ari_values <- sapply(resolutions, function(r) {
#     adjustedRandIndex(clustersResList[[as.character(r)]][,1],
#                       clustersResList[[as.character(r)]][,2])
# })

# plot(resolutions, ari_values, type = "b",
#      xlab = "Resolution", ylab = "ARI",
#      main = "Gene vs Transposon Clustering Similarity")


In [ ]:
TEclusters <- clustersResList[["0.6"]][,2]
table(clustersResList[["0.6"]][,2])

GENEclusters <- clustersResList[["0.5"]][,1]
table(clustersResList[["0.5"]][,1])

In [ ]:
options(repr.plot.width=8, repr.plot.height=7)

concordanceTable <- table(TEclusters, GENEclusters)

rownames(concordanceTable) <- paste0("TE_cluster", rownames(concordanceTable))
colnames(concordanceTable) <- paste0("GENE_cluster", colnames(concordanceTable))

pheatmap(concordanceTable, display_numbers = T,main = "Cluster concordance - TEs VS Genes",
        fontsize_number = 12, fontsize = 12, annotation_names_row = TRUE,
        color = alpha(brewer.pal(9,'Purples')[1:7], 0.5),
        cluster_rows=FALSE, cluster_cols=FALSE, 
         border_color = NA, number_format = "%.0f", cellwidth = 30, cellheight = 30)




In [ ]:
library(clue)
library(grid)
options(repr.plot.width=9, repr.plot.height=8)

# Suppose your matrix has more rows than columns
nr <- nrow(concordanceTable)
nc <- ncol(concordanceTable)

if(nr > nc){
  # pad with zeros to make it square
  M <- cbind(concordanceTable, matrix(0, nrow = nr, ncol = nr - nc))
} else {
  M <- concordanceTable
}

perm <- solve_LSAP(M, maximum = TRUE)

# keep only the original columns
M_reordered <- M[, perm[1:nc]]



ph <- pheatmap(M_reordered[,!is.na(colnames(M_reordered))], 
        display_numbers = T, color = colorRampPalette(brewer.pal(9,'Blues')[1:7])(100),#adjustcolor((colorRampPalette(carto_pal(name="Emrld")))(100),alpha.f=1),
        main = "Cluster concordance - TEs VS Genes",
        cluster_rows=FALSE, cluster_cols=FALSE, 
                fontsize_number = 12, fontsize = 12, annotation_names_row = TRUE, number_color = "black",
         border_color = NA, number_format = "%.0f", cellwidth = 30, cellheight = 30)


# write to PDF safely

pdf(paste0("figures_", dataset_id, "/heatmap_cluster_clusters_concordance_SoloTE.pdf"), width=8, height=9)

grid::grid.newpage()
grid::grid.draw(ph$gtable)

dev.off()


In [ ]:
unique(objGenes_SoloTE$celltype)




In [ ]:
PBMCcelltypeColors <- c("B_cells"="#6D5A5D",
                        "Dendritic Cells"="#C4B3AB", #"#E78063",#"#c492a7",
                        "Monocytes"="#81B29A",
                        "Platelets"="#e78063",
                        "T_cells"="#84B6D6",
                        "Natural Killers"= "#F2CC8F",
                        "Neutrophils"="#c492a7",
                        "Unknown" = "#3D405B")

In [ ]:
#add cell types to TE obj
identical(rownames(objTE_SoloTE@meta.data), 
    rownames(objGenes_SoloTE@meta.data) ) # same cells in same order

objTE_SoloTE$celltype <- objGenes_SoloTE$celltype

In [ ]:
options(repr.plot.width=8, repr.plot.height=3)
concordanceTable <- table(objTE_SoloTE$seurat_clusters, objTE_SoloTE$celltype)

df <- melt(concordanceTable)
df <- df[df$value!=0,]
colnames(df) <- c("cluster","celltype","nCells")
df$cluster <- as.character(df$cluster)
df$clusterSize <- table(objTE_SoloTE$seurat_clusters)[df$cluster]
df$percentage <- as.numeric(df$nCells / df$clusterSize *100)

df$cluster <- factor(df$cluster, levels=c(0,1:length(unique(df$cluster))))


ggplot(df, aes(x=cluster, y=percentage, fill=celltype)) + 
  geom_col() + 
  xlab("TE cluster") +
  scale_fill_manual(values=PBMCcelltypeColors)+
  theme_minimal() + theme(text=element_text(size=18))
ggsave(paste0("figures_", dataset_id,"/clusterIdentity_barplot_SoloTE.pdf"), width=8, height=3)

In [ ]:
options(repr.plot.width=8, repr.plot.height=3)
concordanceTable <- table(objTE_SoloTE$seurat_clusters, objTE_SoloTE$celltype)

df <- melt(concordanceTable)
df <- df[df$value!=0,]
colnames(df) <- c("cluster","celltype","nCells")
df$cluster <- as.character(df$cluster)
df$clusterSize <- table(objTE_SoloTE$seurat_clusters)[df$cluster]
df$percentage <- as.numeric(df$nCells / df$clusterSize *100)

df <- df %>%
  group_by(cluster) %>%
  mutate(main = celltype[which.max(percentage)]) %>%
  ungroup() %>%
  arrange(main, desc(percentage))
df$cluster <- factor(df$cluster, levels = unique(df$cluster))


ggplot(df, aes(x=cluster, y=percentage, fill=celltype)) + 
  geom_col() + 
  xlab("TE cluster") +
  # geom_text(data = sizes,
  #         aes(x = factor(cluster), y = 105, label = clusterSize),
  #         inherit.aes = FALSE) +
  scale_fill_manual(values=PBMCcelltypeColors)+
  theme_minimal() + theme(text=element_text(size=18))
ggsave(paste0("figures_", dataset_id,"/clusterIdentity_perc_barplot_SoloTE_noplatelet.pdf"), width=8, height=3)

In [ ]:
options(repr.plot.width=8, repr.plot.height=3)
concordanceTable <- table(objTE_SoloTE$seurat_clusters, objTE_SoloTE$celltype)

df <- melt(concordanceTable)
df <- df[df$value!=0,]
colnames(df) <- c("cluster","celltype","nCells")
df$cluster <- as.character(df$cluster)
df$clusterSize <- table(objTE_SoloTE$seurat_clusters)[df$cluster]
df$percentage <- as.numeric(df$nCells / df$clusterSize *100)

df <- df %>%
  group_by(cluster) %>%
  mutate(main = celltype[which.max(percentage)]) %>%
  ungroup() %>%
  arrange(main, desc(percentage))
df$cluster <- factor(df$cluster, levels = unique(df$cluster))


ggplot(df, aes(x=cluster, y=nCells, fill=celltype)) + 
  geom_col() + 
  xlab("TE cluster") +
  # geom_text(data = sizes,
  #         aes(x = factor(cluster), y = 105, label = clusterSize),
  #         inherit.aes = FALSE) +
  scale_fill_manual(values=PBMCcelltypeColors)+
  theme_minimal() + theme(text=element_text(size=18))
ggsave(paste0("figures_", dataset_id,"/clusterIdentity_barplot_SoloTE_noplatelet.pdf"), width=8, height=3)

In [ ]:
options(repr.plot.width=8, repr.plot.height=3)
concordanceTable <- table(objTE_SoloTE$seurat_clusters, objTE_SoloTE$celltype)

df <- melt(concordanceTable)
df <- df[df$value!=0,]
colnames(df) <- c("cluster","celltype","nCells")
df$cluster <- as.character(df$cluster)
df$clusterSize <- table(objTE_SoloTE$seurat_clusters)[df$cluster]
df$percentage <- as.numeric(df$nCells / df$clusterSize *100)

df <- df %>%
  group_by(cluster) %>%
  #mutate(main = celltype[which.max(percentage)]) %>%
  #ungroup() %>%
  arrange(desc(clusterSize))
df$cluster <- factor(df$cluster, levels = unique(df$cluster))


ggplot(df, aes(x=cluster, y=nCells, fill=celltype)) + 
  geom_col() + 
  xlab("TE cluster") +
  # geom_text(data = sizes,
  #         aes(x = factor(cluster), y = 105, label = clusterSize),
  #         inherit.aes = FALSE) +
  scale_fill_manual(values=PBMCcelltypeColors)+
  theme_minimal() + theme(text=element_text(size=18))
ggsave(paste0("figures_", dataset_id,"/clusterIdentity_barplot_SoloTE_noplatelet_orderedBySize.pdf"), width=8, height=3)

In [ ]:
options(repr.plot.width=9.5, repr.plot.height=7)

DimPlot(objTE_SoloTE, reduction = "umap", group.by = "celltype",
        cols=PBMCcelltypeColors,
        shuffle=T, pt.size = 0.7) + 
        ggtitle("Gene-derived cell types") +
  theme_pubr() +
  theme(text=element_text(size=20), plot.title = element_text(hjust=0.5)) 
ggsave(paste0("figures_",dataset_id,"/umap_TEs_locus_celltypes_SoloTE.pdf"), device = "pdf", width=9.5, height=7)